In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('ggplot')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (12, 8),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import rateslib as rl
import QuantLib as ql

import datetime
import pytz

NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
LDN_tz = pytz.timezone("Europe/London") 
UTC_tz = pytz.timezone("UTC") 

import sys
sys.path.append("../")

In [2]:
from MDP.IRSwaptions.IRSwaptionMDP import IRSwaptionMDP
from Query.Base.query_resolution import resolve_query
from Query.IRSwaptions import IRSwaptionQuery, IRSwaptionStructure, IRSwaptionValue


In [27]:
# mdp = IRSwaptionMDP(
#     source="GSQUANT-QL",
#     curve_source="ERIS_EOD_LIVE-QL_BASIC",
# )

mdp = IRSwaptionMDP(
    source="MONKEYCUBE-QL",
    curve_source="ERIS_EOD_LIVE-QL_BASIC",
    data_dir=r'C:\Users\chris\clee\ARBS\MDP\IRSwaptions\MONKEYCUBE\YCMONKEY_USD_VOL_CUBE_GAMMA_MIX'
)

In [28]:
as_of = datetime.date(2026, 3, 10)
ctx = mdp.get_pricer(
    {
        "curve_name": "USD-SOFR-1D",
        "timestamp": as_of,
        "ignore_cache": True,
    }
)
ctx

c:\Users\chris\anaconda3\envs\stir\Lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning:

The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified upper bound 0.01. Increasing the bound and calling fit again may find a better value.

c:\Users\chris\anaconda3\envs\stir\Lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning:

The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.



IRSwaptionMarketContext(curve_name='USD-SOFR-1D', as_of_date=datetime.date(2026, 3, 10), curve=QLIRSwapCurve(_ql_curve_id='USD-SOFR-1D', _ql_curve_handle=<QuantLib.QuantLib.YieldTermStructureHandle; proxy of <Swig Object of type 'Handle< YieldTermStructure > *' at 0x000001C87976C5B0> >, _ql_curve_index=<QuantLib.QuantLib.Sofr; proxy of <Swig Object of type 'ext::shared_ptr< Sofr > *' at 0x000001C87976D6F0> >, _meta_data={'timestamp': datetime.date(2026, 3, 10)}), curve_handle=<QuantLib.QuantLib.YieldTermStructureHandle; proxy of <Swig Object of type 'Handle< YieldTermStructure > *' at 0x000001C87976C5B0> >, swap_index=<QuantLib.QuantLib.Sofr; proxy of <Swig Object of type 'ext::shared_ptr< Sofr > *' at 0x000001C87976D6F0> >, vol_handle=<QuantLib.QuantLib.SwaptionVolatilityStructureHandle; proxy of <Swig Object of type 'Handle< SwaptionVolatilityStructure > *' at 0x000001C876700830> >, pricing_engine=<QuantLib.QuantLib.BachelierSwaptionEngine; proxy of <Swig Object of type 'ext::shared_

In [29]:
q = IRSwaptionQuery(
	shorthand="1y1y",
	structure=IRSwaptionStructure.PAYER,
    strike="ATMF+100",
)

q_eff = resolve_query(q, timestamp=as_of, pricer_or_curve=ctx)
package, weights = q_eff.resolve_package(pricer_or_curve=ctx)
vmap = q_eff.build_value_map(pricer_or_curve=ctx, package=package, risk_weights=weights)
float(vmap.apply(IRSwaptionValue.NVOL)), float(vmap.apply(IRSwaptionValue.FWD_PREM))

(89.66695147957718, 5.2622556187243905)

In [30]:
q = IRSwaptionQuery(
	shorthand="1y1y",
	structure=IRSwaptionStructure.RECEIVER,
    strike="ATMF-100",
)

q_eff = resolve_query(q, timestamp=as_of, pricer_or_curve=ctx)
package, weights = q_eff.resolve_package(pricer_or_curve=ctx)
vmap = q_eff.build_value_map(pricer_or_curve=ctx, package=package, risk_weights=weights)
float(vmap.apply(IRSwaptionValue.NVOL)), float(vmap.apply(IRSwaptionValue.FWD_PREM))

(105.62506314712134, 5.262255618724379)